In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.3     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.4     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.0
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
#vars of interest
raw_vars <- read.table('ACE_de_novo_SNVs_Indels_table_Children_included_in_analysis_v2024-03-14.txt', sep = "\t")

# Make the first row as column names
colnames(raw_vars) <- raw_vars[1, ]

# Remove the first row
raw_vars <- raw_vars[-1, ]

#raw_vars$sampleID <- paste0(raw_vars$sampleID, "_", raw_vars$sampleID) 

In [3]:
head(raw_vars)

,variant,sampleID,role,father_sampleID,mother_sampleID,gene,consequence,hgvsc,isPTV,isMIS,isSYN,isIndel,gnomad_non_neuro_AF,gnomad_non_neuro_AF_AFR,MPC,VarKey
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
2,chr1:11847267:C:T,AU3937301,Sibling,AU3937201,AU3937202,NPPA,[missense_variant],ENST00000376480.7:c.296G>A,FALSE,TRUE,FALSE,FALSE,1.46e-05,0,0.023648,chr1:11847267:C:T:AU3937301
3,chr1:13342921:T:C,AU1920301,Sibling,AU1920201,AU1920202,PRAMEF14,[synonymous_variant],ENST00000334600.6:c.1032A>G,FALSE,FALSE,TRUE,FALSE,3.22e-05,0.00040783,NA,chr1:13342921:T:C:AU1920301
4,chr1:13342933:G:T,AU1920301,Sibling,AU1920201,AU1920202,PRAMEF14,[synonymous_variant],ENST00000334600.6:c.1020C>A,FALSE,FALSE,TRUE,FALSE,8.19e-05,0.00011366,NA,chr1:13342933:G:T:AU1920301
5,chr1:13420931:C:T,AU1389305,Sibling,AU1389201,AU1389202,PRAMEF20,[synonymous_variant],ENST00000316412.9:c.1101C>T,FALSE,FALSE,TRUE,FALSE,NA,NA,NA,chr1:13420931:C:T:AU1389305
6,chr1:23971953:C:A,AU4992303,Sibling,AU4992201,AU4992202,SRSF10,[missense_variant],ENST00000492112.2:c.334G>T,FALSE,TRUE,FALSE,FALSE,NA,NA,NA,chr1:23971953:C:A:AU4992303
7,chr1:27997351:C:A,AU1505303,Sibling,AU1505201,AU1505202,EYA3,[missense_variant],ENST00000373871.7:c.1111G>T,FALSE,TRUE,FALSE,FALSE,0,0,1.1554,chr1:27997351:C:A:AU1505303


In [4]:
table(table(raw_vars$variant))


  1 
431 

In [7]:
table(table(raw_vars$variant))


  1 
431 

In [6]:
table(table(raw_vars$sampleID))


 1  2  3  4  5  7 
99 65 37 11  8  1 

In [5]:
length(unique(raw_vars$sampleID))

[1] 221

In [5]:
length(unique(raw_vars$sampleID))

[1] 221

In [6]:
length(unique(raw_vars$variant))

[1] 431

In [7]:
var_indv <- raw_vars %>%
  group_by(variant) %>%
  summarize(sampleIDs = list(unique(sampleID)))

In [8]:
View(var_indv)

variant,sampleIDs
<chr>,<list>
chr10:102872596:C:A,AU4654301
chr10:103999287:A:AC,AU1210301
chr10:119207922:A:T,AU0638302
chr10:16782060:A:C,DG1167-301
chr10:18538287:C:A,AU4524302
chr10:20888208:C:G,AU065606
chr10:49739837:A:G,AU4006301
chr10:50125425:C:T,AU4779302
chr10:63153526:G:C,AU1309301


In [9]:
var_indv[var_indv$variant == "chr1:27997351:C:A",]$sampleIDs

[[1]]
[1] "AU1505303"

In [10]:
table(table(var_indv$variant))


  1 
431 

In [11]:
vars <- as.data.frame(unique(raw_vars[,c("variant")]))
colnames(vars) <- "variant"

In [12]:
head(vars)

,variant
,<chr>
1,chr1:11847267:C:T
2,chr1:13342921:T:C
3,chr1:13342933:G:T
4,chr1:13420931:C:T
5,chr1:23971953:C:A
6,chr1:27997351:C:A


In [13]:
# Split the column into separate columns
split_data <- str_split(vars$variant, ":", simplify = TRUE)

# Rename the columns
colnames(split_data) <- c("chrom", "pos", "ref", "alt")

# Bind the split columns to the original dataframe
vars <- cbind(vars, split_data)

vars$pos <- as.numeric(vars$pos)

vars <- vars[!(vars$chrom == "chrX"),]
# remove x

vars$chr_num <- as.numeric(sub("^chr", "", vars$chrom))

In [14]:
head(vars)

,variant,chrom,pos,ref,alt,chr_num
,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>
1,chr1:11847267:C:T,chr1,11847267,C,T,1
2,chr1:13342921:T:C,chr1,13342921,T,C,1
3,chr1:13342933:G:T,chr1,13342933,G,T,1
4,chr1:13420931:C:T,chr1,13420931,C,T,1
5,chr1:23971953:C:A,chr1,23971953,C,A,1
6,chr1:27997351:C:A,chr1,27997351,C,A,1


In [15]:
find_regions <- function(chr_vars, msp){

# Initialize an empty dataframe to store the results
results_df <- data.frame(variant = character(), zeros = integer(), ones = integer(), stringsAsFactors = FALSE)

# Loop through each variant in chr_vars
for (i in seq_along(chr_vars$variant)) {
  # Subset var_msp for the corresponding position
  var_msp <- msp[msp$spos <= chr_vars$pos[i] & msp$epos >= chr_vars$pos[i], ]

  var_cols <- c(outer(unlist(var_indv[var_indv$variant == chr_vars$variant[i],]$sampleIDs), c(".0", ".1"), paste0))
  #print(var_cols)
  var_msp <- var_msp[,colnames(var_msp) %in% var_cols]
  #print(colnames(var_msp))  
    
 if(nrow(var_msp)>1){
    print(paste("Overlapping variant window:", chr_vars$variant[i]))
  }
  # Extract columns ending with .0
 # id_columns <- grep(paste0(chr_vars$variant[i], "\\.0$"), names(var_msp), value = TRUE)
  id_columns <- grep("\\.0$", names(var_msp), value = TRUE)
  # Initialize counters for zeros and ones
  num_zeros <- 0
  num_ones <- 0
  
  # Loop through each column and count zeros and ones
  for (col_name in id_columns) {
    # Get corresponding .1 column
    id_col_1 <- sub("\\.0$", ".1", col_name)
    
    # Count zeros and ones
    num_zeros <- num_zeros + sum(var_msp[[col_name]] == 0 & var_msp[[id_col_1]] == 0)
    num_ones <- num_ones + sum(var_msp[[col_name]] == 1 & var_msp[[id_col_1]] == 1)
  }
  
  # Append results to the dataframe
  results_df <- rbind(results_df, data.frame(variant = chr_vars$variant[i], zeros = num_zeros, ones = num_ones))
}
 return(results_df)
    }

In [16]:
start.time <- Sys.time()
total_results <- data.frame(variant = character(), zeros = integer(), ones = integer(), stringsAsFactors = FALSE)

msp_dir <- "/u/home/a/afcarrol/project-pasaniuc/Projects/20240423_ace_partial_hm3/rfmix_out/"
#chr <- 1
for(chr in 1:22){
    print(paste("Starting Chromosome:", chr))
    msp_path <- paste0(msp_dir, "chr", chr, ".msp.tsv")

    # Read the first line of the file to extract column names
    col_names <- readLines(msp_path, n = 2)[2]
    col_names <- strsplit(col_names, "\t")[[1]]
    col_names[1] <- substring(col_names[1], 2)  # Remove the leading #

    # Read the .tsv file into R, skipping the first line
    msp <- read.table(msp_path, header = FALSE, sep = "\t", col.names = col_names)
    
    chr_vars <- vars[vars$chr_num == chr,]
    
    chr_result <- find_regions(chr_vars, msp)
    
    total_results <- rbind(total_results, chr_result)
}
end.time <- Sys.time()
time.taken <- round(end.time - start.time,2)
time.taken

[1] "Starting Chromosome: 1"
[1] "Starting Chromosome: 2"
[1] "Starting Chromosome: 3"
[1] "Starting Chromosome: 4"
[1] "Starting Chromosome: 5"
[1] "Starting Chromosome: 6"
[1] "Starting Chromosome: 7"
[1] "Starting Chromosome: 8"
[1] "Starting Chromosome: 9"
[1] "Starting Chromosome: 10"
[1] "Starting Chromosome: 11"
[1] "Starting Chromosome: 12"
[1] "Starting Chromosome: 13"
[1] "Starting Chromosome: 14"
[1] "Starting Chromosome: 15"
[1] "Starting Chromosome: 16"
[1] "Starting Chromosome: 17"
[1] "Starting Chromosome: 18"
[1] "Starting Chromosome: 19"
[1] "Starting Chromosome: 20"
[1] "Starting Chromosome: 21"
[1] "Starting Chromosome: 22"


Time difference of 36.25 secs

In [17]:
head(total_results)

,variant,zeros,ones
,<chr>,<dbl>,<dbl>
1,chr1:11847267:C:T,0,1
2,chr1:13342921:T:C,0,0
3,chr1:13342933:G:T,0,0
4,chr1:13420931:C:T,0,1
5,chr1:23971953:C:A,0,0
6,chr1:27997351:C:A,1,0


In [18]:
table(total_results$zeros)


  0   1 
401  29 

In [19]:
table(total_results$ones)


  0   1 
249 181 